<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 03 · pandas Deep Dive

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
The book needs a bridge between first-table habits and the more realistic
workflows that use multiple files. This notebook gives that bridge a small,
repeatable shape.


This cell reads the dataset into a DataFrame so the later examples can inspect and transform it.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "2_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path

import pandas as pd

data_dir = BOOK_ROOT / 'data'
artifacts_dir = BOOK_ROOT / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(data_dir / 'orders.csv')
customers = pd.read_csv(data_dir / 'customers.csv')

print(orders.head())
print()
print(customers.head())


This cell continues the worked example for the current section.


In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_with_customers = orders.merge(customers, on='customer_id', how='left')

print(orders_with_customers.head())
print()
print(orders_with_customers.dtypes)


This cell groups the data and orders the result so the learner can compare categories clearly.


In [ ]:
country_summary = (
    orders_with_customers
    .groupby('country')['amount']
    .agg(total_amount='sum', average_amount='mean', order_count='size')
    .sort_values('total_amount', ascending=False)
)

category_summary = (
    orders_with_customers
    .pivot_table(
        index='country',
        columns='category',
        values='amount',
        aggfunc='sum',
        fill_value=0,
    )
)

print(country_summary)
print()
print(category_summary)


This cell continues the worked example for the current section.


In [ ]:
clean_orders = (
    orders_with_customers
    .drop_duplicates()
    .assign(
        amount_eur=lambda df: df['amount'].round(2),
        order_month=lambda df: df['order_date'].dt.to_period('M').astype(str),
    )
)

print(clean_orders[['order_id', 'country', 'amount_eur', 'order_month']])


This cell continues the worked example for the current section.


In [ ]:
def format_amount(value: float) -> str:
    return f'{value:,.2f}'

amount_preview = clean_orders['amount'].head().apply(format_amount)

print('formatted amounts:')
print(amount_preview.tolist())

output_path = data_dir / 'orders_clean.csv'
clean_orders.to_csv(output_path, index=False)
print(output_path.resolve())


This cell continues the worked example for the current section.


In [ ]:
summary_path = artifacts_dir / 'pandas_deep_dive_summary.txt'
summary_path.write_text(
    'pandas deep dive summary\n'
    f'Rows: {len(clean_orders):,}\n'
    f'Countries: {", ".join(country_summary.index.tolist())}\n'
    f'Categories: {", ".join(sorted(clean_orders["category"].unique()))}\n'
)

print(summary_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
